# DNABERT-2: Efficient Foundation Model and Benchmark For Multi-Species Genomes
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2306.15006
Generated: 2026-07-12

This notebook checks the environment, installs the project, explains the model, loads the official
pretrained DNABERT-2 (BPE tokenizer + ALiBi encoder), runs a small fine-tuning demo on synthetic
data (no downloads), and shows the paper's reported GUE results for comparison.

**Use a GPU runtime** (Runtime -> Change runtime type -> T4 GPU): DNABERT-2's Flash-Attention code
needs `triton`, which is preinstalled on Colab GPU.

## 1. Environment check

In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU. DNABERT-2 needs triton (GPU) to load. Switch to a T4 runtime.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Installation

In [ ]:
# Install the project. No transformers pin or runtime restart needed — the loader
# patches DNABERT-2's 2023-era incompatibilities (pad_token_id, meta-init, Triton kernel).
import subprocess
r = subprocess.run(["pip","install","-q","-e",".."], capture_output=True, text=True)
print(r.stdout[-800:] if r.returncode==0 else r.stderr[-800:])
import transformers, torch
print("transformers:", transformers.__version__, "| torch:", torch.__version__)

## 3. Paper overview

**Problem.** Genome language models mostly used **k-mer tokenization**, which leaks information
(overlapping) or is sample-inefficient (non-overlapping).

**Key ideas.**
- **BPE tokenization** (Sec 3.1) — variable-length tokens, ~5x shorter sequences, no leakage.
- **ALiBi** (Sec 3.2) — linear attention biases instead of positional embeddings -> arbitrary length.
- **GEGLU** activations + **Flash Attention** + optional **LoRA** fine-tuning.

At ~117M params it matches the 2.5B Nucleotide Transformer (21x fewer params, ~92x less compute) and
introduces **GUE**: 28 datasets / 7 tasks. This repo fine-tunes the official weights on any GUE task.

## 4. Component walkthrough: the BPE tokenizer

DNABERT-2 tokenizes DNA with SentencePiece BPE (vocab 4096) instead of k-mers.

In [ ]:
from src.dnabert2.data.tokenizer import DNATokenizer
tok = DNATokenizer("zhihan1996/DNABERT-2-117M", max_len=32)
print("tokenizer:", tok)
enc = tok.encode_batch(["ACGTACGTACGTTTGG", "TTTTGGGGCCCCAAAA"], max_len=16)
print("input_ids shape:", tuple(enc["input_ids"].shape))
print("example ids:", enc["input_ids"][0].tolist())

## 5. Load the pretrained DNABERT-2 classifier

Loads `zhihan1996/DNABERT-2-117M` via AutoModel (BPE + ALiBi bundled) and attaches a linear head.

In [ ]:
from src.dnabert2.models.classifier import DNABERT2Classifier
model = DNABERT2Classifier.from_pretrained(
    "zhihan1996/DNABERT-2-117M", num_classes=2, pool="mean", device=str(device))
n = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameter count: {n/1e6:.1f}M")

## Mini-training on synthetic data

No downloads: a tiny GC-rich vs AT-rich binary task. Loss should drop within a few steps.

In [ ]:
import random, torch, torch.nn.functional as F
random.seed(0)
def seq(gc): return "".join(random.choice("GCGCAT" if gc else "ATATGC") for _ in range(60))
seqs   = [seq(i%2==0) for i in range(64)]
labels = [0 if i%2==0 else 1 for i in range(64)]

opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
model.train()
for step in range(6):
    batch = seqs[step*8:(step+1)*8] or seqs[:8]
    yb = torch.tensor(labels[step*8:(step+1)*8] or labels[:8]).to(device)
    enc = tok.encode_batch(batch, max_len=64)
    ids, mask = enc["input_ids"].to(device), enc["attention_mask"].to(device)
    opt.zero_grad(); loss = F.cross_entropy(model(ids, mask), yb); loss.backward(); opt.step()
    print(f"step {step} loss {loss.item():.4f}")
print("mini-training complete (loss should trend down)")

## 6. Paper results (GUE) for comparison

In [ ]:
paper_results = {
    "promoter_detection/all (MCC)":      86.77,
    "promoter_detection/notata (MCC)":   94.27,
    "core_promoter_detection/all (MCC)": 69.37,
    "epigenetic_marks/H3 (MCC)":         78.27,
    "tf_human/0 (MCC)":                  71.99,
    "splice/reconstructed (MCC)":        84.99,
    "covid_variant (F1)":                71.02,
}
print("DNABERT-2 reported GUE results (paper Table 4/6):")
for k,v in paper_results.items():
    print(f"  {k:38s} {v}")
print("\nTo reproduce: download a GUE task, then run train.py + evaluate.py.")
print("  python data/download.py --task promoter_detection")
print("  python train.py --config configs/config.yaml")
print("Then feed your metrics back to ArXivist -> Stage 6 Results Comparator.")

## What to do next

1. **Download data (API):** `python data/download.py --task promoter_detection`
2. **Fine-tune:** `python train.py --config configs/config.yaml`  (default: promoter_detection/all)
3. **Other tasks:** `--task epigenetic_marks --subset H3`, etc.
4. **Evaluate:** `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
5. **Compare:** paste your MCC/F1 back to ArXivist (Stage 6).

**Recipe (paper Appendix A.3):** AdamW, lr 3e-5, wd 0.01, batch 32, warmup 50, best-by-val checkpoint.